In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

In [0]:
staging_schema = "stocks.stage"
all_tables = spark.catalog.listTables(staging_schema)

In [0]:
for table in all_tables:
    company_name = table.name
    df = spark.table(f"{staging_schema}.{company_name}")
    df = df.withColumn("company", F.lit(company_name))

    if not spark.catalog.tableExists(f"stocks.bronze.stocks_all"):
        df.write.mode("overwrite").format("delta").saveAsTable("stocks.bronze.stocks_all"
        )

    delta_table = DeltaTable.forName(spark, "stocks.bronze.stocks_all")

    delta_table.alias("t").merge(
        df.alias("s"),
        "s.Date = t.Date and s.company = t.company"
    ).whenNotMatchedInsertAll().execute()